# 01 — Data Audit

**Purpose:** Verify the exact state of the frozen dataset.  
Reproduce verified findings: 48 completed matches, group standings through MD2,  
third-place ranking, name mapping correctness, and data quality summary.

**Freeze date:** 2026-06-23 23:59 UTC  
**Elo snapshot:** 2026-05-27  
**This notebook produces no model inputs — only verification evidence.**

---
**Outputs**
- `outputs/group_standings_freeze.csv`

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path so `src` is importable
PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import hashlib
import pandas as pd

from src.leakage_guard import (
    FREEZE_DATE, ELO_SNAPSHOT_DATE,
    check_freeze_date, check_elo_snapshot, check_canonical_names,
    LeakageError,
)
from src.name_map import CANONICAL_48, TO_CANONICAL, canonicalize, assert_all_canonical
from src.standings import build_group_membership, compute_group_standings, rank_third_placed
from src.features import (
    load_ds2, load_ds4, load_ds6, load_ds8, load_ds10,
    load_ds16, load_ds17, load_ds18, load_ds19,
    load_ds1, load_ds1ext,
)

# ── Archive paths ──────────────────────────────────────────────────────────
DATA_ROOT   = PROJECT_ROOT
ARC_BASE    = DATA_ROOT / "archive.zip"          # DS16, DS17, DS18, DS19
ARC2        = DATA_ROOT / "archive (2).zip"      # DS4, DS10, DS1
ARC3        = DATA_ROOT / "archive (3).zip"      # DS8
ARC4        = DATA_ROOT / "archive (4).zip"      # DS2 (Elo)
ARC6        = DATA_ROOT / "archive (6).zip"      # DS6 (shootouts)
OUTPUTS     = PROJECT_ROOT / "outputs"
OUTPUTS.mkdir(exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Freeze date  : {FREEZE_DATE}")
print(f"Elo snapshot : {ELO_SNAPSHOT_DATE}")

## Cell 2 — Load raw datasets

In [ ]:
ds2  = load_ds2(ARC4)   # Elo ratings snapshot
ds4  = load_ds4(ARC2)   # Competitive match results 2014-2026
ds6  = load_ds6(ARC6)   # Penalty shootout records
ds8  = load_ds8(ARC3)   # World Cup match archive 1930-2022
ds10 = load_ds10(ARC2)  # FIFA rankings
ds16 = load_ds16(ARC_BASE)  # 2026 bracket fixture list
ds17 = load_ds17(ARC_BASE)  # 2026 team roster / group assignments
ds1  = load_ds1(ARC2)       # WC 2026 match results (MD1+MD2)

print(f"DS2  shape: {ds2.shape}   (Elo snapshot)")
print(f"DS4  shape: {ds4.shape}   (competitive matches)")
print(f"DS6  shape: {ds6.shape}   (shootouts)")
print(f"DS8  shape: {ds8.shape}   (WC archive)")
print(f"DS10 shape: {ds10.shape}  (FIFA rankings)")
print(f"DS16 shape: {ds16.shape}  (2026 bracket)")
print(f"DS17 shape: {ds17.shape}  (2026 teams)")
print(f"DS1  shape: {ds1.shape}   (2026 results)")

## Cell 3 — Freeze-date and Elo snapshot leakage checks

In [ ]:
# L1: No DS1/DS4 results after 2026-06-23
check_freeze_date(ds1, date_column="date")
check_freeze_date(ds4, date_column="date")

# L2: Elo snapshot must be 2026-05-27
check_elo_snapshot(ds2)

print("✓ Freeze-date check passed — no post-freeze data in DS1 or DS4")
print("✓ Elo snapshot check passed — snapshot date is 2026-05-27")

## Cell 4 — Frozen 48-match result set

In [ ]:
# DS1 contains all completed WC 2026 matches through 2026-06-23
completed = ds1.dropna(subset=["home_score", "away_score"]).copy()
completed["home_score"] = completed["home_score"].astype(int)
completed["away_score"] = completed["away_score"].astype(int)

assert len(completed) == 48, (
    f"Expected exactly 48 completed matches, got {len(completed)}"
)
print(f"✓ Exactly 48 completed matches confirmed")

# Display the full 48-match table
display_cols = ["date", "home_team", "away_team", "home_score", "away_score"]
available = [c for c in display_cols if c in completed.columns]
completed[available].sort_values("date").reset_index(drop=True)

## Cell 5 — Compute and display group standings

In [ ]:
group_membership = build_group_membership(ds17)

# FIFA ranks for tiebreaker (optional — pass None to use fallback)
fifa_ranks = None
if "fifa_points" in ds10.columns and "country" in ds10.columns:
    fifa_ranks = dict(zip(ds10["country"].map(canonicalize), ds10["fifa_points"]))

standings = compute_group_standings(completed, group_membership, fifa_ranks=fifa_ranks)

print(f"Standings shape: {standings.shape}  (expect 48 rows × 11 cols)")

# Spot-check verified values from the design spec
def _pts(team):
    row = standings[standings["team"] == team]
    assert len(row) == 1, f"{team} not found in standings"
    return int(row["pts"].iloc[0])

for team, expected_pts in [("Mexico", 6), ("France", 6), ("Norway", 6),
                            ("Germany", 6), ("Colombia", 6)]:
    try:
        actual = _pts(team)
        status = "✓" if actual == expected_pts else "✗"
        print(f"{status} {team}: {actual} pts (expected {expected_pts})")
    except AssertionError as e:
        print(f"  SKIP: {e}")

standings

## Cell 6 — Third-place ranking

In [ ]:
ranked_thirds = rank_third_placed(standings, fifa_ranks=fifa_ranks)

assert len(ranked_thirds) == 12, f"Expected 12 third-placed teams, got {len(ranked_thirds)}"
assert ranked_thirds["qualifies_as_third"].sum() == 8, "Exactly 8 thirds should qualify"

print("Third-place ranking (top 8 qualify for R32):")
display_cols = ["team", "group", "pts", "gd", "gf", "third_place_rank", "qualifies_as_third"]
available = [c for c in display_cols if c in ranked_thirds.columns]
ranked_thirds[available].sort_values("third_place_rank").reset_index(drop=True)

## Cell 7 — Name mapping validation

In [ ]:
assert len(CANONICAL_48) == 48, f"Expected 48 canonical teams, got {len(CANONICAL_48)}"

# Verify every canonical name round-trips through canonicalize()
failures = [t for t in CANONICAL_48 if canonicalize(t) != t]
assert not failures, f"Canonical names failed round-trip: {failures}"

# Spot-check known variant mappings
known_variants = {
    "South Korea":           "Korea Republic",
    "Iran":                  "IR Iran",
    "Ivory Coast":           "Côte d'Ivoire",
    "DR Congo":              "Congo DR",
    "Turkey":                "Türkiye",
    "USA":                   "United States",
    "Czech Republic":        "Czechia",
    "Bosnia and Herzegovina":"Bosnia-Herzegovina",
}
variant_fails = {v: canonicalize(v) for v, expected in known_variants.items()
                 if canonicalize(v) != expected}
assert not variant_fails, f"Variant mapping failures: {variant_fails}"

print(f"✓ {len(CANONICAL_48)} canonical teams verified")
print(f"✓ {len(TO_CANONICAL)} variant mappings registered")
print(f"✓ All {len(known_variants)} spot-checked variants correct")

# Display full mapping table
mapping_rows = [{"canonical": t, "sample_variant": next(
    (k for k, v in TO_CANONICAL.items() if v == t), "(identity)")
} for t in sorted(CANONICAL_48)]
pd.DataFrame(mapping_rows)

## Cell 8 — Data quality summary

In [ ]:
quality_rows = []
for name, df in [("DS1 (2026 results)", ds1), ("DS2 (Elo)", ds2),
                 ("DS4 (competitive)", ds4), ("DS6 (shootouts)", ds6),
                 ("DS8 (WC archive)", ds8), ("DS10 (FIFA rank)", ds10)]:
    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]
    quality_rows.append({
        "dataset": name,
        "rows": len(df),
        "cols": len(df.columns),
        "cols_with_nulls": len(cols_with_nulls),
        "total_nulls": int(null_counts.sum()),
        "null_detail": "; ".join(f"{c}:{v}" for c, v in cols_with_nulls.items()) or "none",
    })

quality_df = pd.DataFrame(quality_rows)
print("Data quality summary:")
quality_df

## Cell 9 — Save outputs

In [ ]:
out_path = OUTPUTS / "group_standings_freeze.csv"
standings.to_csv(out_path, index=False)
print(f"Saved → {out_path}  ({len(standings)} rows)")

# Also save ranked thirds
thirds_path = OUTPUTS / "third_place_ranking_freeze.csv"
ranked_thirds.to_csv(thirds_path, index=False)
print(f"Saved → {thirds_path}  ({len(ranked_thirds)} rows)")